In [2]:
import json
import os
from openai import OpenAI
from pydantic import BaseModel, Field

In [3]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [4]:
def search_kb(question: str):
    """
    Load the whole knowledge base from the JSON file.
    (This is a mock function for demonstration purposes, we don't search)
    """
    with open("kb.json", "r") as f:
        return json.load(f)

In [5]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_kb",
            "description": "Get the answer to the user's question from the knowledge base.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {"type": "string"},
                },
                "required": ["question"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    }
]

In [6]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."},
    {"role": "user", "content": "What is the return policy?"},
]

In [7]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant that answers questions from the knowledge base about our e-commerce store.'},
 {'role': 'user', 'content': 'What is the return policy?'}]

In [8]:
completion = client.chat.completions.create(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [9]:
completion

ChatCompletion(id='chatcmpl-CNhhApYcS3srzRR1Y6IRZGOejh1xc', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_aKAufIMDeHtzPJpN9MG6ShtI', function=Function(arguments='{"question":"What is the return policy?"}', name='search_kb'), type='function')]))], created=1759765544, model='gpt-5-nano-2025-08-07', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=29, prompt_tokens=158, total_tokens=187, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))

In [10]:
completion.model_dump()

{'id': 'chatcmpl-CNhhApYcS3srzRR1Y6IRZGOejh1xc',
 'choices': [{'finish_reason': 'tool_calls',
   'index': 0,
   'logprobs': None,
   'message': {'content': None,
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_aKAufIMDeHtzPJpN9MG6ShtI',
      'function': {'arguments': '{"question":"What is the return policy?"}',
       'name': 'search_kb'},
      'type': 'function'}]}}],
 'created': 1759765544,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 29,
  'prompt_tokens': 158,
  'total_tokens': 187,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}}

In [11]:
def call_function(name, args):
    if name == "search_kb":
        return search_kb(**args)


for tool_call in completion.choices[0].message.tool_calls:
    name = tool_call.function.name
    args = json.loads(tool_call.function.arguments)
    messages.append(completion.choices[0].message)

    result = call_function(name, args)
    messages.append(
        {"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)}
    )

In [12]:
messages

[{'role': 'system',
  'content': 'You are a helpful assistant that answers questions from the knowledge base about our e-commerce store.'},
 {'role': 'user', 'content': 'What is the return policy?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_aKAufIMDeHtzPJpN9MG6ShtI', function=Function(arguments='{"question":"What is the return policy?"}', name='search_kb'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_aKAufIMDeHtzPJpN9MG6ShtI',
  'content': '{"records": [{"id": 1, "question": "What is the return policy?", "answer": "Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days."}, {"id": 2, "question": "Do you ship internationally?", "answer": "Yes, we ship to over 50 countries worldwide. International shipping typically takes 7-14 business days 

In [14]:
class KBResponse(BaseModel):
    answer: str = Field(description="The answer to the user's question.")
    source: int = Field(description="The record id of the answer.")


completion_2 = client.beta.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
    response_format=KBResponse
)

In [22]:
completion_2.model_dump()

{'id': 'chatcmpl-CNhiYKkKdNJZGmaRJLr7m1Mxucfhk',
 'choices': [{'finish_reason': 'stop',
   'index': 0,
   'logprobs': None,
   'message': {'content': '{"answer":"Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.","source":1}',
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': None,
    'parsed': {'answer': 'Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.',
     'source': 1}}}],
 'created': 1759765630,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 436,
  'prompt_tokens': 414,
  'total_tokens': 850,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'rea

In [15]:
final_response = completion_2.choices[0].message.parsed
print(final_response.answer)
print(final_response.source)

Items can be returned within 30 days of purchase with original receipt. Refunds will be processed to the original payment method within 5-7 business days.
1


In [16]:
messages = [
    {"role": "system", "content": "You are a helpful assistant that answers questions from the knowledge base about our e-commerce store."},
    {"role": "user", "content": "What is the weather in Tokyo?"},
]

completion_3 = client.beta.chat.completions.parse(
    model="gpt-5-nano",
    messages=messages,
    tools=tools,
)

In [19]:
completion_3.model_dump()

{'id': 'chatcmpl-CNhjYwgzXmlubkn9HtVJc3tKJuPMG',
 'choices': [{'finish_reason': 'tool_calls',
   'index': 0,
   'logprobs': None,
   'message': {'content': None,
    'refusal': None,
    'role': 'assistant',
    'annotations': [],
    'audio': None,
    'function_call': None,
    'tool_calls': [{'id': 'call_THkMSODNUkJe0XWTkTcNEwQx',
      'function': {'arguments': '{"question":"What is the weather in Tokyo?"}',
       'name': 'search_kb',
       'parsed_arguments': {'question': 'What is the weather in Tokyo?'}},
      'type': 'function'}],
    'parsed': None}}],
 'created': 1759765692,
 'model': 'gpt-5-nano-2025-08-07',
 'object': 'chat.completion',
 'service_tier': 'default',
 'system_fingerprint': None,
 'usage': {'completion_tokens': 478,
  'prompt_tokens': 159,
  'total_tokens': 637,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 448,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, '

In [21]:
print(completion_3.choices[0].message.content)

None
